# Importación de datos

En este cuadernillo se describe el proceso de importación de datos que se han utilizado en el proyecto. Se han obtenido datos desde dos fuentes movieLens y TMDB. 

## MovieLens


Los datos de movielens se han descargado de https://grouplens.org/datasets/movielens/. Se ha utilizado la versión de 100K que está compuesta por 4 archivos csv. 

## TMDB

Los datos de TMDB se han recogido haciendo llamadas a su API (https://developer.themoviedb.org/reference/intro/getting-started).   

Para realizar estas peticiones es necesario obtener un token (registrado en el archivo .env) que se carga a continuación.

In [1]:
import os

In [2]:
token = os.getenv('TMDB_TOKEN')

Cabeceras de la API

In [3]:
headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {token}"
}

Las llamadas a la API se hacen utilizando el identificador proporcionado por movieLens almacenado en el fichero links.csv.

Lectura de datos y obtención de los identificadores (se filtran para que no haya repetidos)

In [4]:
import pandas as pd

In [5]:
datos = pd.read_csv("../data/01_raw/movieLens/links.csv")
moviesId = datos.sort_values('tmdbId')['tmdbId'].dropna().drop_duplicates().astype(int).tolist()

Se crea un archivo de log para registrar las peticiones que han fallado. Se almacena en la carpeta logs

In [6]:
import logging

In [7]:
logging.basicConfig(filename="../logs/requestTMDB.log",
                    level=logging.WARNING,
                    format="%(asctime)s | %(levelname)s | %(message)s")

In [8]:
import time
import json
import requests

Se crea la función llamadaTMDB que realiza la petición HTTP mediante el endpoint movie/{id} y devuelve la respuesta en formato json. 

In [9]:
def llamadaTMDB (id):
    url = f"https://api.themoviedb.org/3/movie/{id}"
    try:
        response = requests.get(url, headers=headers, timeout=10) 
        if response.status_code != 200:
            logging.warning(f"tmdbId = {id} | HTTP_Status = {response.status_code}")
            return None
        return response.json()
    except requests.exceptions.RequestException as error:
        logging.error(f"tmdbId = {id} | exception = {error}") 
        return None

La función descargaPeliculas recorre toda la lista de identificadores, utiliza la función anterior para extraer los datos y si la petición fue resuelta con éxito guarda el registro en el directorio ./data/01_raw/TMDB.

In [10]:
def descargaPeliculas():
    for id in moviesId:
        data = llamadaTMDB(id)
        if data is None:    
            print(f"Error en la película de tmdbId: {id}")
            continue
        archivo = os.path.join("../data/01_raw/TMDB", f"movie_{id}.json")
        with open(archivo, 'w') as json_file:
            json.dump(data, json_file)
        time.sleep(0.03)


Descarga de datos: 

In [ ]:
descargaPeliculas()

Error en la película de tmdbId: 876
Error en la película de tmdbId: 2670
Error en la película de tmdbId: 6075
Error en la película de tmdbId: 7096
Error en la película de tmdbId: 8677
Error en la película de tmdbId: 9795
Error en la película de tmdbId: 10700
